# RNNs on Mathematical Equations

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that we're on Python 3 and that the hardware accelerator is set to "GPU".

In [ ]:
!nvidia-smi

## Installing and Importing PyTorch Lightning and the Other Required Libraries

In [ ]:
!pip install -q lightning torchmetrics

In [ ]:
import operator

import lightning
import matplotlib.pyplot as plt
import numpy
import sklearn.model_selection
import torch
import torchmetrics
from lightning.pytorch.callbacks import Callback
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## Defining the Vocabulary

In this lab, we'll define equations such as `3 + 1 = 4`. The model will have to predict `4` given `3 + 1` as input.

To start, let's create the vocabulary:

In [ ]:
operations = list("+*-/")
numbers = list("0123456789.")
padding = [" "]

index_to_char = numbers + operations + padding
char_to_index = {c: i for i, c in enumerate(index_to_char)}

print(f"Index to character: {index_to_char}")
print(f"Character to index: {char_to_index}")

## Using the Vocabulary to Encode and Decode Equations

We can now use this vocabulary to turn text equations into sequences of numbers that a neural network can understand:

In [ ]:
def encode(characters: str) -> torch.Tensor:
  return torch.tensor([char_to_index[char] for char in characters])


def decode(array: torch.Tensor) -> str:
  return ''.join(index_to_char[int(i)] for i in array)

*Test the `encode` and `decode` functions. Do you think the neural network will be able to work directly on the output of `encode`, or will additional preprocessing be needed?*

In [ ]:
# Your test code here

Your answer here

### Solution

In [ ]:
equation = "3/4+2-5"
result = "0"
print("─" * 50)
print("Equation")
print("─" * 50)
print(f"Raw form          {equation}")
print(f"Encoding          {encode(equation)}")
print(f"Encoding/decoding {decode(encode(equation))}")
print()
print("─" * 50)
print("Result")
print("─" * 50)
print(f"Raw form          {result}")
print(f"Encoding          {encode(result)}")
print(f"Encoding/decoding {decode(encode(result))}")

This encoding isn't enough for neural network processing: you'll need to either one-hot encode on top of it, or use an embedding layer.

## Generating Data

We can now generate examples, which will be used for training, validation and testing.

To do so, we'll randomly pick one of the 4 defined operations and generate random integers to apply it to. This process is repeated until we reach the desired number of examples.

In [ ]:
def make_maths_problem(n_samples: int = 1000,
                       n_digits: int = 3,
                       invert: bool = True
                       ) -> tuple[torch.Tensor, torch.Tensor]:
  equations = []
  results = []
  seen = set()

  math_operation = {"+": operator.add,
                    "-": operator.sub,
                    "*": operator.mul,
                    "/": operator.truediv}

  # Maximum length of the string describing the operation
  max_equation_len = 2 * n_digits + 1
  max_result_len = 2 * n_digits

  while len(equations) < n_samples:
    # Pick a random operation
    operation = numpy.random.choice(operations)

    # Generate two integers within the size limit
    left, right = numpy.random.randint(10 ** n_digits, size=2)

    equation = f"{left}{operation}{right}"

    if equation not in seen:
      seen.add(equation)

      # Compute the result. Since left and right are numpy integers,
      # this computation raises no exception, even on division by 0
      math_result = math_operation[operation](left, right)

      # Start over if the result isn't usable
      if math_result == numpy.inf or numpy.isnan(math_result):
        continue

      # The result can be very long (0.3333333…): limit its length
      result = str(math_result)[:max_result_len]

      # "Pad" so that all sequences have the same length
      padded_equation = equation.ljust(max_equation_len)
      padded_result = result.ljust(max_result_len)

      # Invert if the invert argument is given
      if invert:
        padded_equation = padded_equation[::-1]

      equations.append(padded_equation)
      results.append(padded_result)

  X = torch.stack(list(map(encode, equations)))
  y = torch.stack(list(map(encode, results)))
  return X, y

Let's test this function with a dozen examples:

In [ ]:
X, y = make_maths_problem(10, n_digits=3)

print(f"X shape:  {tuple(X.shape)}")
print(f"y shape:  {tuple(y.shape)}")

print()
print("A few generated examples")
for encoded_equation, encoded_result in zip(X, y):
  # Equation inversion is enabled by default: undo it
  # to display the original equation
  equation = decode(encoded_equation.flip(0))
  result = decode(encoded_result)
  print(f"{equation} = {result}")

We can now create the dataset and the necessary splits:

In [ ]:
X, y = make_maths_problem(100_000, n_digits=3)
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, train_size=0.8)

## Defining the Model

To process these character sequences, we'll use an RNN. As said before, the input sequences will need to be one-hot encoded or passed through an embedding layer. Here, we'll use the embedding layer.

In [ ]:
class SequencePredictor(lightning.LightningModule):
  """Lightning wrapper: one class is predicted for each output character."""

  def __init__(self,
               model: nn.Module,
               learning_rate: float = 1e-3,
               n_digits: int = 3,
               num_classes: int = len(index_to_char)) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # An example input lets Lightning display the shapes of the input and output
    # tensors of each layer in the model summary
    self.example_input_array = torch.zeros(1, 2 * n_digits + 1,
                                           dtype=torch.long)
    # torchmetrics needs one metric instance per stage
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val")
    })

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    return self.model(equations)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    equations, results = batch
    # The logits have shape (batch, length, vocabulary), but the loss and the
    # metric expect the class dimension in second position
    logits = self(equations).transpose(1, 2)
    loss = nn.functional.cross_entropy(logits, results)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, results)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)


@torch.no_grad()
def predict(model: nn.Module,
            X: torch.Tensor,
            batch_size: int = 1024) -> torch.Tensor:
  """Apply a model to X, batch by batch, leaving its mode unchanged."""
  device = next(model.parameters()).device
  was_training = model.training
  model.eval()
  logits = torch.cat([model(batch.to(device)).cpu()
                      for batch in X.split(batch_size)])
  model.train(was_training)
  return logits

In [ ]:
class Seq2Seq(nn.Module):
  """Encode the equation, then decode the result character by character."""

  def __init__(self,
               hidden_size: int = 1024,
               n_digits: int = 3,
               embedding_dim: int = 16) -> None:
    super().__init__()
    self.result_len = 2 * n_digits

    # Encoder: two stacked GRU layers, the second one returning only its
    # last hidden state
    self.embedding = nn.Embedding(len(index_to_char), embedding_dim)
    self.encoder = nn.GRU(embedding_dim,
                          hidden_size,
                          num_layers=2,
                          batch_first=True)

    # Decoder
    self.decoder = nn.GRU(hidden_size, hidden_size, batch_first=True)

    # The dense output layer is applied at each timestep: nn.Linear
    # only works on the last dimension of the tensor
    self.head = nn.Linear(hidden_size, len(index_to_char))

  def forward(self, equations: torch.Tensor) -> torch.Tensor:
    embedded = self.embedding(equations)
    _, hidden = self.encoder(embedded)

    # The encoder's last hidden state summarizes the whole equation. We duplicate it
    # as many times as we want output characters: it's a simple way to
    # condition the decoding of the result on the encoding of the equation
    summary = hidden[-1].unsqueeze(1).repeat(1, self.result_len, 1)

    decoded, _ = self.decoder(summary)
    return self.head(decoded)


def rnn_model(hidden_size: int = 1024, n_digits: int = 3) -> nn.Module:
  return Seq2Seq(hidden_size, n_digits)


model = SequencePredictor(rnn_model())

## Training the Model

We'll regularly display predictions on fixed examples to see how the model evolves.

In [ ]:
# We'll use the same examples at each evaluation to see the model
# progress
n_test = 20
indexes = torch.randperm(X_test.shape[0])[:n_test]
X_examples, y_examples = X_test[indexes], y_test[indexes]

In [ ]:
def examples(model: nn.Module) -> None:
  guesses = predict(model, X_examples).argmax(dim=-1)
  for encoded_equation, encoded_result, encoded_guess in zip(
      X_examples, y_examples, guesses):
    equation = decode(encoded_equation.flip(0))
    result = decode(encoded_result)
    guess = decode(encoded_guess)
    print(f"{equation} = {result} ?= {guess}")


class PrintExamples(Callback):
  """Show the model's guesses on fixed examples every few epochs."""

  def __init__(self, every_n_epochs: int = 10) -> None:
    super().__init__()
    self.every_n_epochs = every_n_epochs

  def on_validation_epoch_end(self,
                              trainer: lightning.Trainer,
                              pl_module: lightning.LightningModule) -> None:
    epoch = trainer.current_epoch + 1
    if epoch % self.every_n_epochs:
      return
    print()
    print("─" * 50)
    print(f"After {epoch} epochs:")
    examples(pl_module)


batch_size = 4096
train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=batch_size,
                          shuffle=True)
val_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=batch_size)

trainer = lightning.Trainer(max_epochs=100,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="seq2seq"),
                            enable_checkpointing=False,
                            callbacks=[PrintExamples(every_n_epochs=10)])
trainer.fit(model, train_loader, val_loader)

## Evaluating the Model

To evaluate our model, we'll compute 3 things on the test set:

- the illegal predictions (when the model's output can't be interpreted as a float)
- the accuracy
- the fractional error ($\frac{y_{pred} - y}{y}$)

In [ ]:
def evaluate(model: nn.Module) -> None:
  # Compute the model's predictions
  predictions = predict(model, X_test).argmax(dim=-1)

  # Decode the model's predictions into strings
  str_predictions = [decode(prediction) for prediction in predictions]

  # Decode the model's predictions into floats
  # Sometimes the emitted string doesn't represent a valid float
  # Create a boolean mask to filter these elements afterwards
  illegal_predictions_mask = torch.zeros(X_test.shape[0], dtype=torch.bool)
  float_predictions = []
  for i, str_prediction in enumerate(str_predictions):
    try:
      float_prediction = float(str_prediction)
      float_predictions.append(float_prediction)
    except ValueError:
      illegal_predictions_mask[i] = True
  illegal_ratio = int(illegal_predictions_mask.sum()) / X_test.shape[0]
  print(f"Percentage of illegal predictions: {illegal_ratio * 100}%")

  y_pred = torch.tensor(float_predictions)
  y = torch.tensor([float(decode(result)) for result in y_test])
  y = y[~illegal_predictions_mask]

  accuracy = int((y == y_pred).sum()) / X_test.shape[0]
  print(f"Accuracy: {accuracy:.2f}")

  y_denominator = y.clone()
  y_denominator[y == 0] = 1

  fractional_difference = (y_pred - y) / y_denominator

  print(f"Mean fractional error: {fractional_difference.mean()}")

  plt.hist(fractional_difference.numpy(), bins=1000)
  plt.title("Global view of the distribution of fractional errors")
  plt.xlabel("Fractional error")
  plt.ylabel("Example count")
  plt.xscale("symlog")
  plt.yscale("log")
  plt.show()

  plt.hist(fractional_difference.numpy(), bins=20, range=(-0.1, 0.1))
  plt.title("Zoomed view around 0 of the distribution of fractional errors")
  plt.xlabel("Fractional error")
  plt.ylabel("Example count")
  plt.show()


evaluate(model)

## Success per Operation

Does the model handle each of the 4 operations equally well? For each operation, compute the share of test equations whose predicted result is **entirely** correct, and explain the differences.

In [ ]:
# Your code here

### Solution

In [ ]:
predictions = predict(model, X_test).argmax(dim=-1)
# A result is only correct if all its characters are
exact = (predictions == y_test).all(dim=1)
equations = [decode(encoded_equation.flip(0)) for encoded_equation in X_test]
for operation in operations:
  mask = torch.tensor([operation in equation for equation in equations])
  print(f"{operation}: {float(exact[mask].float().mean()) * 100:.1f}% of "
        "exact results")

Usually, addition and subtraction work best: each digit of the result only depends on a few digits of the operands and a carry. In a multiplication, each digit of the result depends on all those of the operands, and a division often gives decimals with no simple structure: these two operations are much harder.

## Should the Equation Be Reversed?

`make_maths_problem` reverses the equations by default (`invert=True`): the encoder reads `321+654` as `456+123`. Generate a dataset without reversal, train the same model on it for 30 epochs, and compare its validation accuracy curve with the first model's. How can the gap be explained?

In [ ]:
# Your code here

### Solution

In [ ]:
import pathlib

import pandas


def val_accuracy(trainer: lightning.Trainer) -> pandas.Series:
  metrics = pandas.read_csv(pathlib.Path(trainer.logger.log_dir) / "metrics.csv")
  return metrics.dropna(subset=["val_accuracy"]).set_index("epoch")["val_accuracy"]


X_raw, y_raw = make_maths_problem(100_000, n_digits=3, invert=False)
X_raw_train, X_raw_test, y_raw_train, y_raw_test = (
    sklearn.model_selection.train_test_split(X_raw, y_raw, train_size=0.8))

raw_model = SequencePredictor(rnn_model())
raw_trainer = lightning.Trainer(max_epochs=30,
                                accelerator="auto",
                                devices=1,
                                logger=CSVLogger("logs", name="seq2seq_raw"),
                                enable_checkpointing=False)
raw_trainer.fit(
    raw_model,
    DataLoader(TensorDataset(X_raw_train, y_raw_train),
               batch_size=batch_size,
               shuffle=True),
    DataLoader(TensorDataset(X_raw_test, y_raw_test), batch_size=batch_size))

val_accuracy(trainer)[:30].plot(label="Reversed equation")
val_accuracy(raw_trainer).plot(label="Equation in order")
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy (per character)")
plt.legend()
plt.show()

Usually, the model learns faster on reversed equations. The first characters of the result mostly depend on the first digits of the operands, especially those of the left operand. Reversed, the equation starts with the right operand and ends with the left one: the encoder reads those first digits last, so they're more present in its last hidden state, which is all the decoder gets. It's the trick of the paper [*Sequence to Sequence Learning with Neural Networks*](https://arxiv.org/abs/1409.3215), which reversed the source sentences in machine translation.